In [ ]:
from huggingface_hub import whoami

username = whoami()["name"]

In [ ]:
MODELS_DIR = "./models"

In [ ]:
import os
from huggingface_hub import create_repo, upload_folder
from merge_lora_to_hf import merge_lora_to_hf

import shutil
import os

In [ ]:
def infer_base_model(run_name: str) -> str:
    if run_name.startswith("OLMo-2-0425-1B"):
        return "allenai/OLMo-2-0425-1B"
    if run_name.startswith("Llama-3.2-1B"):
        return "meta-llama/Llama-3.2-1B"
    if run_name.startswith("Qwen2.5-0.5B"):
        return "Qwen/Qwen2.5-0.5B"

    raise ValueError(f"Unknown base model: {run_name}")


def upload_lora_only(adapter_dir: str, repo_id: str, private: bool):
    create_repo(repo_id=repo_id, private=private, exist_ok=True)
    
    upload_folder(
        folder_path=adapter_dir,
        repo_id=repo_id,
        repo_type="model",
        allow_patterns=[
            "adapter_model.*",
            "adapter_config.json",
            "experiment_config.json",
            "trainer_state.json",
            "optimizer.pt",
            "*.jinja",
            "README.md",
            "checkpoint-*/*",
            "checkpoint-*",
        ],
    )

    print(f"Uploaded LoRA → https://huggingface.co/{repo_id}")



In [ ]:
runs = sorted(os.listdir(MODELS_DIR))
lora_runs = [
    r for r in runs
    if not r.endswith("-merged")
    and os.path.isdir(os.path.join(MODELS_DIR, r))
]

for run_name in lora_runs:
    adapter_dir = os.path.join(MODELS_DIR, run_name)
    merged_dir = os.path.join(MODELS_DIR, run_name + "-merged")

    print(f"\n=== Processing {run_name} ===")

    base_model = infer_base_model(run_name)

    lora_repo = f"{username}/{run_name}"
    merged_repo = f"{username}/{run_name}-merged"
    
    
    src = "./chat_template.jinja"
    dst = os.path.join(adapter_dir, "chat_template.jinja")

    if os.path.exists(src):
        shutil.copy(src, dst)
        
    else:
        raise FileNotFoundError(f"{src} not found")
    upload_lora_only(
        adapter_dir=adapter_dir,
        repo_id=lora_repo,
        private=True,
    )
    
    

